In [1]:
import time
import requests


def get_json(url):
    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0", "Accept": "application/json"},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def save_report(lines, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(line + "\n")
    print("گزارش ذخیره شد:", filename)


print("ابزارها آماده‌اند.")

ابزارها آماده‌اند.


## مرحله ۱: سبد نمونه

In [2]:
portfolio = [
    {"symbol": "فولاد", "shares": 10000, "buy_price": 3900},
    {"symbol": "فملی",  "shares": 2000,  "buy_price": 26500},
    {"symbol": "وبملت", "shares": 30000, "buy_price": 1350},
]
buy_fee = 0.003712    # کارمزد خرید
sell_fee = 0.0088   # کارمزد فروش با مالیات
risk_free = 0.39   # سود بی‌ریسک 

## https://learning.emofid.com/trading-fees/
## https://irfarabi.com/fixed-incom-fund

total_invested = 0
for p in portfolio:
    total_invested += p["shares"] * p["buy_price"]

print(f"سرمایه اولیه: {total_invested:,} ریال")
print(len(portfolio), "سهم در سبد")

سرمایه اولیه: 132,500,000 ریال
3 سهم در سبد


## مرحله ۲: گرفتن داده tsetmc (۶ دارایی)

In [3]:
tsetmc_symbols = {
    "فولاد":  "فولاد",
    "فملی":   "فملی",
    "وبملت":  "وبملت",
    "کهربا":  "GOLD_FUND",
    "اهرم":   "LEVERAGE",
    "کمند":   "FIXED_INCOME",
}

fallback_codes = {
    "فولاد": "46348559193224090",
    "فملی":  "35425587644337450",
    "وبملت": "778253364357513",
    "عیار":  "34144395039913458",
    "کهربا": "25559236668122210",
    "اهرم":  "17914401175772326",
    "کمند":  "34718633636164421",
}


def find_ins_code(symbol):
    try:
        items = get_json("https://cdn.tsetmc.com/api/Instrument/GetInstrumentSearch/" + symbol)["instrumentSearch"]
    except Exception as e:
        print("  جست‌وجو خطا:", e)
        return None
    for item in items:
        name = item["lVal18AFC"].replace("ي", "ی").replace("ك", "ک").strip()
        if name == symbol:
            return item["insCode"]
    return None


def fetch_history(code):
    url = f"https://cdn.tsetmc.com/api/ClosingPrice/GetClosingPriceDailyList/{code}/250"
    return get_json(url)["closingPriceDaily"]


def build_small(rows):
    small = {}
    for row in rows:
        if row["pClosing"] > 0:
            small[row["dEven"]] = row["pClosing"]
    return small


prices = {}
tsetmc_codes = {}

# پاس اول: جست‌وجو، اگر نشد پشتیبان
for symbol, name in tsetmc_symbols.items():
    print("\n" + name)
    rows = None
    code = find_ins_code(symbol)
    time.sleep(1)

    if code is not None:
        try:
            rows = fetch_history(code)
            print("  با کد جست‌وجو:", code)
        except Exception as e:
            print("  خطا در تاریخچه:", e)
            rows = None

    if rows is None and symbol in fallback_codes:
        time.sleep(1)
        backup = fallback_codes[symbol]
        try:
            rows = fetch_history(backup)
            code = backup
            print("  با کد پشتیبان:", backup)
        except Exception as e:
            print("  پشتیبان هم خطا:", e)

    if rows is None:
        print("  ❌ رد شد")
        continue

    small = build_small(rows)
    prices[name] = small
    tsetmc_codes[name] = code
    print(f"  ✅ {len(small)} روز، آخرین: {max(small)}")
    time.sleep(1)


# پاس دوم: هر نمادی که افتاد، دوباره با کد پشتیبان
missing = [sym for sym, name in tsetmc_symbols.items() if name not in prices]

if missing:
    print("\n--- پاس نجات ---")
    for symbol in missing:
        name = tsetmc_symbols[symbol]
        if symbol not in fallback_codes:
            print(f"{name}: کد پشتیبان ندارد")
            continue
        print(f"\n{name}")
        time.sleep(2)
        try:
            rows = fetch_history(fallback_codes[symbol])
            small = build_small(rows)
            prices[name] = small
            tsetmc_codes[name] = fallback_codes[symbol]
            print(f"  ✅ {len(small)} روز")
        except Exception as e:
            print("  ❌ باز هم خطا:", e)


print("\n--- خلاصه ---")
for name, d in prices.items():
    print(f"{name}: {len(d)} روز")


فولاد
  با کد جست‌وجو: 46348559193224090
  ✅ 250 روز، آخرین: 20261111

فملی
  با کد جست‌وجو: 35425587644337450
  ✅ 250 روز، آخرین: 20261111

وبملت
  با کد جست‌وجو: 778253364357513
  ✅ 250 روز، آخرین: 20261111

GOLD_FUND
  با کد جست‌وجو: 25559236668122210
  ✅ 250 روز، آخرین: 20261111

LEVERAGE
  با کد جست‌وجو: 17914401175772326
  ✅ 250 روز، آخرین: 20261111

FIXED_INCOME
  با کد جست‌وجو: 34718633636164421
  ✅ 250 روز، آخرین: 20261111

--- خلاصه ---
فولاد: 250 روز
فملی: 250 روز
وبملت: 250 روز
GOLD_FUND: 250 روز
LEVERAGE: 250 روز
FIXED_INCOME: 250 روز


## مرحله ۳: گرفتن داده tgju (۵ دارایی)

In [4]:
tgju_keys = {
    "DOLLAR":  "price_dollar_rl",
    "COIN":    "sekee",
    "GOLD18":  "geram18",
    "TETHER":  "crypto-tether-irr",
    "BITCOIN": "crypto-bitcoin-irr",
}

for name, key in tgju_keys.items():
    print("\n" + name)
    try:
        data = get_json(f"https://api.tgju.org/v1/market/indicator/summary-table-data/{key}?length=250")
        rows = data["data"]
    except Exception as e:
        print("  خطا:", e)
        continue

    small = {}
    for row in rows:
        try:
            d = int(row[6].replace("/", ""))
            p = float(row[3].replace(",", ""))
            if p > 0:
                small[d] = p
        except Exception:
            continue

    prices[name] = small
    print(f"  ✅ {len(small)} روز، آخرین: {max(small) if small else '—'}")

print(f"\nمجموع دارایی‌ها: {len(prices)}")


DOLLAR
  ✅ 250 روز، آخرین: 20261112

COIN
  ✅ 250 روز، آخرین: 20261112

GOLD18
  ✅ 250 روز، آخرین: 20261112

TETHER
  ✅ 250 روز، آخرین: 20261113

BITCOIN
  ✅ 250 روز، آخرین: 20261113

مجموع دارایی‌ها: 11


## مرحله ۴: روزهای مشترک (۱۲۰ روز آخر)

In [5]:
common = None
for name in prices:
    s = set(prices[name])
    if common is None:
        common = s
    else:
        common = common & s

days = sorted(common)[-120:]

print(f"{len(days)} روز مشترک، از {days[0]} تا {days[-1]}")

120 روز مشترک، از 20260518 تا 20261111


## مرحله ۵: بازده روزانه + اصلاح تله افزایش سرمایه

In [6]:
tsetmc_names = ["فولاد", "فملی", "وبملت", "GOLD_FUND", "LEVERAGE", "FIXED_INCOME"]

returns = {}

for name in prices:
    # اصلاح فقط روی tsetmc
    if name in tsetmc_names:
        for i in range(1, len(days)):
            today = prices[name][days[i]]
            yest  = prices[name][days[i - 1]]
            x = today / yest - 1
            if x < -0.15:
                print("افت مصنوعی:", name, days[i], round(x * 100, 1), "%")
                ratio = today / yest
                for j in range(i):
                    prices[name][days[j]] *= ratio

    # بازده
    r = []
    for i in range(1, len(days)):
        x = prices[name][days[i]] / prices[name][days[i - 1]] - 1
        r.append(x)
    returns[name] = r

first_key = list(returns)[0]
print(f"\n{len(returns)} دارایی، هر کدام {len(returns[first_key])} بازده")

افت مصنوعی: فملی 20260801 -25.8 %

11 دارایی، هر کدام 119 بازده


## مرحله ۶: ارزش سبد و افزودن MY_PORTFOLIO

In [7]:
missing = [p["symbol"] for p in portfolio if p["symbol"] not in prices]

if missing:
    print("⚠️ این نمادها در prices نیستند:", missing)
    print("کلیدهای موجود:", list(prices.keys()))
else:
    value = {}
    for t in days:
        total = 0
        for p in portfolio:
            total += p["shares"] * prices[p["symbol"]][t]
        value[t] = total

    prices["MY_PORTFOLIO"] = value

    r = []
    for i in range(1, len(days)):
        r.append(value[days[i]] / value[days[i - 1]] - 1)
    returns["MY_PORTFOLIO"] = r

    print(f"اول دوره: {int(value[days[0]]):,} ریال | آخر دوره: {int(value[days[-1]]):,} ریال")
    print(f"تعداد دارایی‌ها: {len(returns)}")

اول دوره: 72,433,016 ریال | آخر دوره: 152,180,000 ریال
تعداد دارایی‌ها: 12


## مرحله ۷: شش معیار در یک حلقه

In [8]:
scores = []

for name in returns:
    r = returns[name]

    first = prices[name][days[0]]
    last  = prices[name][days[-1]]
    total_return = last / first - 1

    mean = sum(r) / len(r)

    var = 0
    for x in r:
        var += (x - mean) ** 2
    std = (var / (len(r) - 1)) ** 0.5

    ann_return = mean * 240
    ann_risk   = std * 240 ** 0.5

    sharpe = (ann_return - risk_free) / ann_risk if ann_risk > 0 else 0

    down = 0
    for x in r:
        if x < 0:
            down += x ** 2
    down_risk = (down / len(r)) ** 0.5 * 240 ** 0.5
    sortino = (ann_return - risk_free) / down_risk if down_risk > 0 else 0

    peak = first
    max_dd = 0
    for t in days:
        p = prices[name][t]
        if p > peak:
            peak = p
        dd = p / peak - 1
        if dd < max_dd:
            max_dd = dd

    scores.append({
        "name":    name,
        "total":   total_return,
        "risk":    ann_risk,
        "sharpe":  sharpe,
        "sortino": sortino,
        "max_dd":  max_dd,
    })

print(f"{len(scores)} دارایی حساب شد")

12 دارایی حساب شد


## مرحله ۸: سود خالص بعد از کارمزد

In [9]:
total_cost  = 0
total_value = 0
rows = []

for p in portfolio:
    price_now = prices[p["symbol"]][days[-1]]

    cost = p["shares"] * p["buy_price"] * (1 + buy_fee)
    val  = p["shares"] * price_now * (1 - sell_fee)

    rows.append((p["symbol"], price_now, cost, val, val - cost))

    total_cost  += cost
    total_value += val

profit = total_value - total_cost

print("سود هر سهم (تومان):")
for symbol, price_now, cost, val, pr in rows:
    print(f"  {symbol}: {pr / 10:+,.0f}")

print(f"\nسود خالص: {profit / 10:+,.0f} تومان ({profit / total_cost * 100:+.1f}%)")

سود هر سهم (تومان):
  فولاد: +991,963
  فملی: +338,096
  وبملت: +454,838

سود خالص: +1,784,898 تومان (+13.4%)


## مرحله ۹: جدول رتبه‌بندی بر اساس شارپ


In [10]:
pairs = []
for s in scores:
    pairs.append((-s["sharpe"], s["name"]))

table = ["RANK  ASSET          RETURN%  RISK%  SHARPE  SORTINO  MAX_DD%"]

rank = 1
for neg, name in sorted(pairs):
    for s in scores:
        if s["name"] == name:
            row  = f"{rank:<5} {name:<13} {s['total']*100:8.1f} {s['risk']*100:7.1f}"
            row += f" {s['sharpe']:7.2f} {s['sortino']:8.2f} {s['max_dd']*100:8.1f}"
            table.append(row)
            rank += 1

for line in table:
    print(line)

RANK  ASSET          RETURN%  RISK%  SHARPE  SORTINO  MAX_DD%
1     فملی             180.9    35.5    4.97     8.76    -19.3
2     LEVERAGE         233.1    48.6    4.46     7.69    -23.1
3     MY_PORTFOLIO     110.1    26.5    4.32     7.61    -14.1
4     فولاد             90.1    29.2    3.26     5.35    -14.8
5     وبملت             74.9    39.2    2.08     3.65    -23.7
6     TETHER            48.3    28.2    1.58     2.42    -13.5
7     DOLLAR            46.8    27.2    1.55     2.53    -13.9
8     BITCOIN           43.4    47.8    0.94     1.62    -26.8
9     COIN              33.5    31.6    0.77     1.21    -18.3
10    GOLD18            32.6    31.7    0.72     1.12    -18.6
11    GOLD_FUND         26.9    30.2    0.45     0.70    -17.9
12    FIXED_INCOME      -0.5     8.8   -4.51    -4.66     -2.7


## مرحله ۱۰: پیشنهادها + ذخیره گزارش


In [11]:
by_name = {}
for s in scores:
    by_name[s["name"]] = s

notes = []

# تمرکز بیش از ۴۰٪
for symbol, price_now, cost, val, pr in rows:
    if val / total_value > 0.40:
        notes.append(f"REBALANCE: {symbol} بیش از 40% سبد است")

my = by_name["MY_PORTFOLIO"]

if my["sharpe"] < by_name["GOLD_FUND"]["sharpe"]:
    notes.append("CONSIDER: بخشی از سبد به طلا منتقل شود")

if my["max_dd"] < -0.20:
    notes.append("RISK: افت سبد از -20% بدتر بوده")

# هم‌جهتی با دلار
same = 0
mine   = returns["MY_PORTFOLIO"]
dollar = returns["DOLLAR"]
n = min(len(mine), len(dollar))
for i in range(n):
    if (mine[i] > 0) == (dollar[i] > 0):
        same += 1
co = same / n * 100
notes.append(f"CO_DIRECTION: هم‌جهتی سبد با دلار {co:.0f}% روزها")

# گزارش
report = []
report.append("=" * 60)
report.append("REPORT P3 — کارنامه پورتفوی")
report.append("=" * 60)
report.append(f"Period: {days[0]} to {days[-1]}")
report.append(f"Days:   {len(days)}")
report.append(f"Assets: {len(scores)}")
report.append("")
report.extend(table)
report.append("")
report.append(f"NET PROFIT: {profit / 10:+,.0f} تومان ({profit / total_cost * 100:+.1f}%)")
report.append("")
report.append("RECOMMENDATIONS:")
for note in notes:
    report.append("  " + note)

save_report(report, "report.txt")

print("\nپیشنهادها:")
for note in notes:
    print("  -", note)

گزارش ذخیره شد: report.txt

پیشنهادها:
  - CO_DIRECTION: هم‌جهتی سبد با دلار 45% روزها


## (اختیاری)  استراتژی‌های امتیازی


In [12]:
# ۱) متعادل‌سازی
print("=== متعادل‌سازی ===")
target = total_value / len(rows)
print(f"هدف هر سهم: {target:,.0f} ریال")
for symbol, price_now, cost, val, pr in rows:
    diff = target - val
    shares = int(diff / price_now)
    print(f"  {symbol}: {shares:+,} سهم")

# ۲) وزن معکوس ریسک
print("\n=== وزن معکوس ریسک ===")
pick = ["MY_PORTFOLIO", "GOLD_FUND", "DOLLAR"]
inv = {}
for n in pick:
    inv[n] = 1 / by_name[n]["risk"]
tot = sum(inv.values())
for n in pick:
    print(f"  {n}: {inv[n] / tot:.0%}")

# ۳) مومنتوم ۶۰ روزه
print("\n=== مومنتوم ۶۰ روزه ===")
mom = []
for name in returns:
    r60 = prices[name][days[-1]] / prices[name][days[-61]] - 1
    mom.append((-r60, name))
for neg, name in sorted(mom)[:2]:
    print(f"  {name}: {-neg:+.1%}")

# ۴) حد ضرر
print("\n=== افت فعلی سبد ===")
v = prices["MY_PORTFOLIO"]
peak = max(v.values())
dd_now = v[days[-1]] / peak - 1
print(f"  افت از سقف: {dd_now:.1%}")
if dd_now < -0.15:
    print("  ⚠️ هشدار: افت بیش از ۱۵٪")

# ۵) بک‌تست سبد دائمی
print("\n=== سبد دائمی ===")
four = ["MY_PORTFOLIO", "GOLD_FUND", "DOLLAR", "FIXED_INCOME"]
combo = []
for i in range(len(days) - 1):
    s = 0
    for name in four:
        s += returns[name][i]
    combo.append(s / 4)

returns["PERMANENT"] = combo
val = 1
prices["PERMANENT"] = {days[0]: 1}
for i in range(len(combo)):
    val *= 1 + combo[i]
    prices["PERMANENT"][days[i + 1]] = val

print("  PERMANENT اضافه شد. سلول‌های ۸ و ۱۰ را دوباره بزن.")

=== متعادل‌سازی ===
هدف هر سهم: 50,280,272 ریال
  فولاد: +245 سهم
  فملی: -220 سهم
  وبملت: +3,343 سهم

=== وزن معکوس ریسک ===
  MY_PORTFOLIO: 35%
  GOLD_FUND: 31%
  DOLLAR: 34%

=== مومنتوم ۶۰ روزه ===
  فولاد: +105.3%
  LEVERAGE: +79.4%

=== افت فعلی سبد ===
  افت از سقف: -3.2%

=== سبد دائمی ===
  PERMANENT اضافه شد. سلول‌های ۸ و ۱۰ را دوباره بزن.
